In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

df = pd.read_csv('dataset/short_form_brain_rot_cognitive_decay.csv')

# Filter untuk membuang gender 'Non-Binary'
df = df[df['gender'] != 'Non-Binary'].copy()

num_cols = ['age', 'daily_short_form_hours', 'avg_clip_duration_sec',
            'swipes_per_minute', 'scroll_sessions_day', 'bedtime_scroll_min',
            'total_sleep_hours', 'multitasking_score', 'total_screen_time_hours']
cat_cols = ['gender', 'primary_platform']
target = 'cognitive_decay_tier'

In [2]:
# split dengan data yang sudah difilter (tanpa Non-Binary), y masih string
X = df[num_cols + cat_cols]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_test = X_train.copy(), X_test.copy()

In [3]:
# 1. Outlier: clip pakai batas IQR dari train
for c in num_cols:
    q1, q3 = X_train[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    X_train[c] = X_train[c].clip(low, high)
    X_test[c] = X_test[c].clip(low, high)

In [4]:
# 2. Encoding target (ordinal)
urutan = {'Normal / Intact Focus': 0,
          'Mild Attentional Strain': 1,
          'Severe Focus Fragmentation': 2,
          'Acute Brain Rot / Executive Burnout': 3}
y_train = y_train.map(urutan)
y_test = y_test.map(urutan)

In [5]:
# 3. Transformasi Fitur (Scaling & Encoding) final untuk X
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols)
    ]
)

In [6]:
# Fit dan transform X_train, lalu transform X_test
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [7]:
# Kembalikan ke bentuk DataFrame agar mudah dibaca 
nama_fitur = preprocessor.get_feature_names_out()
X_train_final = pd.DataFrame(X_train_processed, columns=nama_fitur, index=X_train.index)
X_test_final = pd.DataFrame(X_test_processed, columns=nama_fitur, index=X_test.index)

In [8]:
# Menampilkan hasil akhir preprocessing
print("Dimensi X_train_final:", X_train_final.shape)
print("Dimensi X_test_final:", X_test_final.shape)

Dimensi X_train_final: (6986, 13)
Dimensi X_test_final: (1747, 13)


In [9]:
# Melihat 5 baris pertama dari data latih yang sudah diproses
display(X_train_final.head())

,num__age,num__daily_short_form_hours,num__avg_clip_duration_sec,num__swipes_per_minute,num__scroll_sessions_day,num__bedtime_scroll_min,num__total_sleep_hours,num__multitasking_score,num__total_screen_time_hours,cat__gender_Male,cat__primary_platform_Snapchat Spotlight,cat__primary_platform_TikTok,cat__primary_platform_YouTube Shorts
7787,1.710168,0.681126,0.587865,-0.707369,0.959297,0.873325,-1.221138,0.547003,0.444403,0.0,0.0,1.0,0.0
8981,0.236760,0.071827,-0.224920,-0.141666,-0.171099,-0.306338,-0.897279,-0.175625,-0.324392,1.0,0.0,1.0,0.0
4066,-1.026161,-0.925207,0.170489,-0.443374,-1.113096,-0.418687,-0.681373,-1.207951,-1.246947,0.0,0.0,0.0,1.0
2261,0.131517,-1.589896,2.147532,-1.235358,-1.489894,-1.008518,0.614061,-1.207951,-1.451960,1.0,0.0,0.0,1.0
6506,0.762977,0.182609,-0.005248,-0.330234,0.205699,-0.222076,0.182250,0.082456,0.598163,0.0,0.0,1.0,0.0


In [10]:
# Melihat ringkasan statistik (rata-rata, min, max) setelah di-scale
display(X_train_final.describe())

,num__age,num__daily_short_form_hours,num__avg_clip_duration_sec,num__swipes_per_minute,num__scroll_sessions_day,num__bedtime_scroll_min,num__total_sleep_hours,num__multitasking_score,num__total_screen_time_hours,cat__gender_Male,cat__primary_platform_Snapchat Spotlight,cat__primary_platform_TikTok,cat__primary_platform_YouTube Shorts
count,6.986000e+03,6.986000e+03,6.986000e+03,6.986000e+03,6.986000e+03,6.986000e+03,6.986000e+03,6.986000e+03,6.986000e+03,6986.000000,6986.000000,6986.000000,6986.000000
mean,1.548528e-16,-3.814107e-17,2.298635e-16,4.688809e-16,7.869774e-17,6.407700e-17,5.710990e-16,-8.950438e-17,-1.266284e-16,0.509018,0.074578,0.406957,0.178786
std,1.000072e+00,1.000072e+00,1.000072e+00,1.000072e+00,1.000072e+00,1.000072e+00,1.000072e+00,1.000072e+00,1.000072e+00,0.499954,0.262728,0.491302,0.383201
min,-1.341892e+00,-1.589896e+00,-1.872455e+00,-1.499353e+00,-1.489894e+00,-1.654523e+00,-2.786454e+00,-1.207951e+00,-2.066996e+00,0.000000,0.000000,0.000000,0.000000
25%,-8.156744e-01,-7.590346e-01,-7.301638e-01,-7.450824e-01,-7.362972e-01,-7.557330e-01,-6.813733e-01,-8.466371e-01,-7.344169e-01,0.000000,0.000000,0.000000,0.000000
50%,-1.842137e-01,-2.051267e-01,-2.721523e-02,-2.925201e-01,-1.710993e-01,-1.659018e-01,7.429679e-02,-1.240091e-01,-1.706332e-01,1.000000,0.000000,0.000000,0.000000
75%,6.577339e-01,5.703444e-01,6.537662e-01,4.617505e-01,5.824980e-01,5.924526e-01,7.220140e-01,6.502351e-01,5.981627e-01,1.000000,0.000000,1.000000,0.000000
max,2.762603e+00,2.564413e+00,2.729661e+00,2.272000e+00,2.560691e+00,2.614731e+00,2.557213e+00,2.895543e+00,2.597032e+00,1.000000,1.000000,1.000000,1.000000


In [11]:
print(y_train.value_counts())
print(y_test.value_counts())

cognitive_decay_tier
0    2998
1    2857
2    1049
3      82
Name: count, dtype: int64
cognitive_decay_tier
0    750
1    715
2    262
3     20
Name: count, dtype: int64
